<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/monolayer_heterostructure1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reference: https://journals.aps.org/prmaterials/abstract/10.1103/PhysRevMaterials.7.014009

In [ ]:
!pip install -q jarvis-tools ase

In [ ]:
!wget https://figshare.com/ndownloader/files/39756997 -O monolayer_data.json

In [ ]:
from jarvis.db.jsonutils import loadjson



jdata = loadjson("monolayer_data.json")

In [ ]:
len(jdata)

In [ ]:
from jarvis.core.atoms import Atoms
import numpy as np
import matplotlib.pyplot as plt
from math import floor
%matplotlib inline

#Select 2 JARVIS-DFT IDs for monolayers
jid1='JVASP-664'
jid2='JVASP-652'
def band_alignment_diagram(vbms=[], cbms=[], labels=[]):
    x = np.arange(len(vbms)) + 0.5
    emin = floor(min(vbms)) - 1.0

    fig = plt.figure(figsize=(10, 8))

    plt.rcParams.update({"font.size": 22})
    plt.bar(x, np.array(vbms) - emin, bottom=emin, width=0.2)
    plt.bar(x, -np.array(cbms), bottom=cbms, width=0.2)
    plt.xlim(0, len(labels))
    plt.ylim(emin, 0)
    plt.xticks(x, labels)
    # plt.labels(labels, rotation=90)
    plt.axhline(y=-4.5, linestyle="-.")
    plt.axhline(y=-5.73, linestyle="-.")
    plt.text(1, -4, "${H^+}/{H_2}$")
    plt.text(1, -6.2, "${O_2}/{H_2O}$")
    # plt.title("2D: Positions of VBM and CBM")
    plt.ylabel("Energy relative to vacuum [eV]")
    plt.show()


def get_hetero_type(A={}, B={}):
    stack = "na"
    int_type = "na"
    vbm_a1 = A["scf_vbm"] - A["avg_max"]
    vbm_b1 = B["scf_vbm"] - B["avg_max"]
    cbm_a1 = A["scf_cbm"] - A["avg_max"]
    cbm_b1 = B["scf_cbm"] - B["avg_max"]
    try:
        # if A['phi']>B['phi']:
        if A["scf_vbm"] - A["avg_max"] < B["scf_vbm"] - B["avg_max"]:
            stack = "BA"
            print("NOT-SWAPPED")
        else:
            C = A
            D = B
            A = D
            B = C
            stack = "AB"
            print("SWAPPED")
            # tmp=B
            # B=A
            # A=tmp
        vbm_a = A["scf_vbm"] - A["avg_max"]
        vbm_b = B["scf_vbm"] - B["avg_max"]
        cbm_a = A["scf_cbm"] - A["avg_max"]
        cbm_b = B["scf_cbm"] - B["avg_max"]
        #  print ('vbm_a,vbm_b,cbm_b,cbm_a',vbm_a,vbm_b,cbm_b,cbm_a)
        if vbm_a < vbm_b and vbm_b < cbm_b and cbm_b < cbm_a:
            int_type = "I"
        elif vbm_a < vbm_b and vbm_b < cbm_a and cbm_a < cbm_b:
            int_type = "II"
        elif vbm_a < cbm_a and cbm_a < vbm_b and vbm_b < cbm_b:
            int_type = "III"
    except:
        pass
    return int_type, stack, vbm_a1, vbm_b1, cbm_a1, cbm_b1



for i in jdata:
    if i["jid"] == jid1:
        strt1 = Atoms.from_dict(i["atoms"])
        ase_strt1 = strt1.ase_converter()
        phiA = i["phi"]
        break
for i in jdata:
    if i["jid"] == jid2:
        strt2 = Atoms.from_dict(i["atoms"])
        ase_strt2 = strt2.ase_converter()
        phiB = i["phi"]
        break
if strt1 != None and strt2 != None:
    print('Formula',strt1.composition.reduced_formula,strt2.composition.reduced_formula)
    (
        int_type,
        stack,
        vbm_a,
        vbm_b,
        cbm_a,
        cbm_b,
    ) = get_hetero_type(A=phiA, B=phiB)
    vbms = [vbm_b, vbm_a]
    cbms = [cbm_b, cbm_a]

    labels = [
        strt2.composition.reduced_formula,
        strt1.composition.reduced_formula,
    ]
    label2=strt2.composition.reduced_formula+str("(")+str(jid2)+str(")")
    label1=strt1.composition.reduced_formula+str("(")+str(jid1)+str(")")
    plot_url = band_alignment_diagram(
        vbms=vbms, cbms=cbms, labels=labels
    )


In [ ]:
jdata[0]

In [ ]:
jdata[10]

In [ ]:
from jarvis.db.figshare import data

dft_2d = data('dft_2d')

In [ ]:
import pandas as pd

df1=pd.DataFrame(dft_2d)
df2=pd.DataFrame(jdata)

In [ ]:
df3=pd.merge(df1,df2,on='jid')

In [ ]:
df3[['jid','formula','exfoliation_energy','phi']]

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.hist(df3[df3['exfoliation_energy']!='na']['exfoliation_energy'])
plt.show()